# Fase 1 — Fine-Tuning e Otimização com Optuna do Classificador Multi-Tarefa (Kingdomino)

Este notebook executa no **Google Colab Pro (GPU)** toda a **Fase 1 de Fine-Tuning e Otimização de Hiperparâmetros (Optuna)** do classificador multi-tarefa de células do Kingdomino:
- **`logits_terreno` (8 classes):** `castelo`, `deserto`, `floresta`, `mineracao`, `oceano`, `pantano`, `planicie`, `vazio`
- **`logits_coroas` (4 classes):** `0`, `1`, `2`, `3` coroas

### Estrutura do Notebook
1. **Configuração do Ambiente e Verificação de GPU:** instalação de pacotes, montagem do Google Drive, localização automática dos arquivos do projeto e ativação de precisão mista (`torch.amp.autocast`).
2. **Dataset e Augmentation Físico Direcionado aos Erros do Baseline (`C0`):**
   - *Simulação de Oclusão por Torre 3D do Castelo* (25% das amostras elegíveis de treino);
   - *Simulação de Reflexo Especular / Flash* (20% das amostras de treino);
   - *Variações Geométricas e de Luz* (rotações `0°, 90°, 180°, 270°`, deslocamento `±6 px`, brilho/contraste `±15%`, matiz fixa `0`).
3. **Balanceamento de Classes:** `WeightedRandomSampler` no `DataLoader` de treino + `CrossEntropyLoss` com pesos inversos à frequência e `label_smoothing` ajustável.
4. **Estudo Automatizado com Optuna (30 trials):** maximização da métrica composta na validação (`0.40 * F1_macro_terreno + 0.45 * F1_macro_coroas + 0.15 * Taxa_Tabuleiros_100_Perfeitos`) explorando `mobilenet_v3_small` (a partir do checkpoint atual), `efficientnet_b0` e `convnext_atto` (`<= 20 MB` em FP32).
5. **Exportação ONNX FP32 e Gráficos Coloridos de Auditoria (`resultados_fase1/`):** exportação de `melhor_classificador_fase1_fp32.onnx`, geração dos 4 painéis gráficos em `300 dpi` e salvamento de `relatorio_fase1.json`.

## 1. Configuração do Ambiente e Verificação de GPU
Instala as bibliotecas necessárias no ambiente do Colab Pro (`optuna`, `timm`, `onnx`, `onnxruntime`, `seaborn`, `scikit-learn`) e configura o diretório raiz do projeto e a GPU com precisão mista (`torch.amp.autocast`).

In [ ]:
!pip install -q optuna timm onnx onnxruntime seaborn scikit-learn matplotlib opencv-python-headless

In [ ]:
import copy
import csv
import json
import math
import os
import random
import sys
import time
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
import optuna
import seaborn as sns
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.transforms import functional as TF

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.INFO)


def configurar_ambiente_projeto():
    em_colab = "google.colab" in sys.modules
    if em_colab:
        try:
            from google.colab import drive
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive")
        except Exception as erro:
            print(f"Aviso ao montar Google Drive: {erro}")

    candidatos_raiz = [
        os.path.abspath("."),
        os.path.abspath(".."),
        "/content/pontuador_de_kingdomino",
        "/content/drive/MyDrive/pontuador_de_kingdomino",
        "/content/drive/MyDrive/Desenvolvimento/pontuador_de_kingdomino",
    ]

    raiz_encontrada = os.path.abspath(".")
    for candidato in candidatos_raiz:
        caminho_csv_teste = os.path.join(
            candidato, "dados", "dataset_celulas", "rotulos_celulas.csv"
        )
        if os.path.isfile(caminho_csv_teste):
            raiz_encontrada = os.path.abspath(candidato)
            break

    if raiz_encontrada not in sys.path:
        sys.path.insert(0, raiz_encontrada)

    return raiz_encontrada


def fixar_sementes(semente=42):
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semente)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


SEMENTE_GLOBAL = 42
fixar_sementes(SEMENTE_GLOBAL)

RAIZ_PROJETO = configurar_ambiente_projeto()
PASTA_DADOS = os.path.join(RAIZ_PROJETO, "dados")
PASTA_CELULAS = os.path.join(PASTA_DADOS, "celulas_recortadas")
CAMINHO_CSV_CELULAS = os.path.join(PASTA_DADOS, "dataset_celulas", "rotulos_celulas.csv")
CAMINHO_MANIFESTO_V2 = os.path.join(PASTA_DADOS, "manifesto_dataset_v2.json")
CAMINHO_BASELINE_JSON = os.path.join(PASTA_DADOS, "metricas_baseline_c0_d0.json")
CAMINHO_CHECKPOINT_C0 = os.path.join(RAIZ_PROJETO, "modelos", "classificador_celulas.pth")
PASTA_RESULTADOS_FASE1 = os.path.join(RAIZ_PROJETO, "resultados_fase1")
os.makedirs(PASTA_RESULTADOS_FASE1, exist_ok=True)

TEM_GPU = torch.cuda.is_available()
DISPOSITIVO = torch.device("cuda" if TEM_GPU else "cpu")
TIPO_AMP = "cuda" if TEM_GPU else "cpu"
USAR_AMP = TEM_GPU

print("=" * 78)
print("CONFIGURAÇÃO DO AMBIENTE — FASE 1 (CLASSIFICADOR MULTI-TAREFA)")
print("=" * 78)
print(f"Raiz do projeto:          {RAIZ_PROJETO}")
print(f"Dispositivo selecionado:  {DISPOSITIVO}")
if TEM_GPU:
    nome_gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU detectada:            {nome_gpu} ({vram_gb:.2f} GB VRAM)")
    print("Precisão mista (AMP):     HABILITADA (torch.amp.autocast)")
else:
    print("GPU detectada:            Nenhuma (executando em CPU)")
print(f"Pasta de resultados:      {PASTA_RESULTADOS_FASE1}")
print("=" * 78)

In [ ]:
CLASSES_TERRENO = [
    "castelo",
    "deserto",
    "floresta",
    "mineracao",
    "oceano",
    "pantano",
    "planicie",
    "vazio",
]
CLASSES_COROAS = [0, 1, 2, 3]

MAPA_TERRENO_IDX = {nome: idx for idx, nome in enumerate(CLASSES_TERRENO)}
MAPA_COROAS_IDX = {int(c): idx for idx, c in enumerate(CLASSES_COROAS)}

MEDIA_IMAGENET = [0.485, 0.456, 0.406]
DESVIO_IMAGENET = [0.229, 0.224, 0.225]
TAMANHO_IMAGEM = 160
LIMITE_TAMANHO_FP32_MB = 20.0

with open(CAMINHO_BASELINE_JSON, "r", encoding="utf-8") as arq:
    METRICAS_BASELINE = json.load(arq)

BASELINE_VAL_C0 = METRICAS_BASELINE["validacao"]["classificador_c0"]
SCORE_COMPOSTO_BASELINE_C0 = (
    0.40 * float(BASELINE_VAL_C0["f1_macro_terreno"])
    + 0.45 * float(BASELINE_VAL_C0["f1_macro_coroas"])
    + 0.15 * (float(BASELINE_VAL_C0["taxa_tabuleiros_perfeitos"]) / 100.0)
)

print("MÉTRICAS OFICIAIS DO BASELINE (C0) NA VALIDAÇÃO:")
print(f"  - Acurácia Terreno (8 classes):       {BASELINE_VAL_C0['acuracia_terreno_pct']:.2f}%")
print(f"  - F1-Macro Terreno (8 classes):       {BASELINE_VAL_C0['f1_macro_terreno']:.4f}")
print(f"  - Acurácia Coroas  (4 classes):       {BASELINE_VAL_C0['acuracia_coroas_pct']:.2f}%")
print(f"  - F1-Macro Coroas  (4 classes):       {BASELINE_VAL_C0['f1_macro_coroas']:.4f}")
print(f"  - Acurácia Célula Exata:              {BASELINE_VAL_C0['acuracia_celula_exata_pct']:.2f}%")
print(f"  - Taxa de Tabuleiros 100% Perfeitos:  {BASELINE_VAL_C0['taxa_tabuleiros_perfeitos']:.2f}% ({BASELINE_VAL_C0['tabuleiros_100_perfeitos']}/{BASELINE_VAL_C0['total_tabuleiros_avaliados']})")
print(f"  - Score Composto Baseline C0:         {SCORE_COMPOSTO_BASELINE_C0:.4f}")

## 2. Dataset e Augmentation Físico Direcionado aos Erros do Baseline
Carrega os registros saneados de `dados/dataset_celulas/rotulos_celulas.csv` mantendo a divisão estrita por tabuleiro (`treino` vs `validacao`) definida no manifesto e no Baseline `C0`.

### Augmentations Físicos Implementados:
1. **Simulação de Oclusão por Torre 3D do Castelo (`aplicar_oclusao_torre_castelo`):** em **25%** das amostras de treino (exceto `castelo` e `vazio`), sobrepõe na borda inferior da célula (ocupando entre **20% e 35%** da altura) um recorte real extraído de células de `castelo` pertencentes exclusivamente ao conjunto de treino, ensinando a rede a focar no terreno visível ao redor da torre 3D.
2. **Simulação de Reflexo Especular / Flash (`aplicar_reflexo_especular_flash`):** em **20%** das amostras de treino, aplica uma mancha elíptica suave de brilho intenso para evitar que peças estouradas pelo flash sejam confundidas com `vazio`.
3. **Variações Geométricas e de Luz (`aplicar_variacoes_geometricas_e_luz`):** rotações ortogonais aleatórias (`0°, 90°, 180°, 270°`), deslocamento aleatório de recorte (`±6 px`) e variação moderada de brilho/contraste (`±15%`), mantendo a matiz (`hue`) estritamente fixa em `0`.

In [ ]:
def carregar_divisao_tabuleiros_oficial(caminho_manifesto, metricas_baseline):
    mapa_conjunto = {}
    if os.path.isfile(caminho_manifesto):
        with open(caminho_manifesto, "r", encoding="utf-8") as arq:
            manifesto = json.load(arq)
        for prefixo, info in manifesto.get("tabuleiros", {}).items():
            mapa_conjunto[prefixo] = info.get("conjunto", "treino")
    else:
        val_c0 = metricas_baseline.get("validacao", {}).get("classificador_c0", {})
        prefixos_val = set(val_c0.get("lista_tabuleiros_perfeitos", []))
        for item in val_c0.get("lista_tabuleiros_com_erro", []):
            prefixos_val.add(item["prefixo"])
        for pref in prefixos_val:
            mapa_conjunto[pref] = "validacao"
    return mapa_conjunto


def carregar_amostras_saneadas(caminho_csv, pasta_celulas, mapa_conjunto):
    amostras_treino = []
    amostras_validacao = []

    with open(caminho_csv, "r", encoding="utf-8") as arq:
        leitor = csv.DictReader(arq)
        for linha in leitor:
            nome_arq = linha["arquivo"]
            caminho_img = os.path.join(pasta_celulas, nome_arq)
            if not os.path.isfile(caminho_img):
                continue

            prefixo = linha["prefixo"]
            conjunto = linha.get("conjunto") or mapa_conjunto.get(prefixo, "treino")
            registro = {
                "arquivo": nome_arq,
                "caminho": caminho_img,
                "prefixo": prefixo,
                "linha": int(linha["linha"]),
                "coluna": int(linha["coluna"]),
                "codigo_original": linha["codigo_original"],
                "terreno": linha["terreno"],
                "coroas": int(linha["coroas"]),
                "classe_combinada": linha["classe_combinada"],
                "conjunto": conjunto,
            }
            if conjunto == "validacao":
                amostras_validacao.append(registro)
            else:
                amostras_treino.append(registro)

    tab_treino = {r["prefixo"] for r in amostras_treino}
    tab_val = {r["prefixo"] for r in amostras_validacao}
    intersecao = tab_treino.intersection(tab_val)
    if intersecao:
        raise RuntimeError(f"Vazamento entre treino e validação detectado: {intersecao}")

    return amostras_treino, amostras_validacao


def extrair_banco_recortes_castelo_treino(amostras_treino, tamanho_padrao=160):
    recortes_castelo = []
    for item in amostras_treino:
        if item["terreno"] != "castelo":
            continue
        img_bgr = cv2.imread(item["caminho"])
        if img_bgr is None or img_bgr.size == 0:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (tamanho_padrao, tamanho_padrao), interpolation=cv2.INTER_AREA)
        recortes_castelo.append(img_rgb)
    return recortes_castelo


MAPA_CONJUNTO_TABULEIROS = carregar_divisao_tabuleiros_oficial(
    CAMINHO_MANIFESTO_V2, METRICAS_BASELINE
)
AMOSTRAS_TREINO, AMOSTRAS_VALIDACAO = carregar_amostras_saneadas(
    CAMINHO_CSV_CELULAS, PASTA_CELULAS, MAPA_CONJUNTO_TABULEIROS
)
BANCO_CASTELOS_TREINO = extrair_banco_recortes_castelo_treino(
    AMOSTRAS_TREINO, tamanho_padrao=TAMANHO_IMAGEM
)

print(f"Tabuleiros de Treino:     {len({r['prefixo'] for r in AMOSTRAS_TREINO})} ({len(AMOSTRAS_TREINO)} células)")
print(f"Tabuleiros de Validação:  {len({r['prefixo'] for r in AMOSTRAS_VALIDACAO})} ({len(AMOSTRAS_VALIDACAO)} células)")
print(f"Recortes reais de castelo (banco de oclusão 3D do treino): {len(BANCO_CASTELOS_TREINO)}")

In [ ]:
def aplicar_deslocamento_aleatorio_recorte(imagem_rgb, deslocamento_max_px=6):
    if deslocamento_max_px <= 0:
        return imagem_rgb
    altura, largura = imagem_rgb.shape[:2]
    dx = random.randint(-deslocamento_max_px, deslocamento_max_px)
    dy = random.randint(-deslocamento_max_px, deslocamento_max_px)
    if dx == 0 and dy == 0:
        return imagem_rgb

    acolchoada = cv2.copyMakeBorder(
        imagem_rgb,
        deslocamento_max_px,
        deslocamento_max_px,
        deslocamento_max_px,
        deslocamento_max_px,
        cv2.BORDER_REFLECT_101,
    )
    y0 = deslocamento_max_px + dy
    x0 = deslocamento_max_px + dx
    return acolchoada[y0 : y0 + altura, x0 : x0 + largura]


def aplicar_oclusao_torre_castelo(
    imagem_rgb,
    terreno,
    banco_castelos,
    probabilidade=0.25,
    fracao_altura_min=0.20,
    fracao_altura_max=0.35,
):
    """
    Em 25% das amostras de treino (exceto castelo e vazio), sobrepõe na borda inferior
    da célula (ocupando 20% a 35% da altura) um recorte real extraído de células de castelo
    do conjunto de treino, simulando a invasão da torre 3D sobre células vizinhas.
    """
    if terreno in ("castelo", "vazio") or not banco_castelos:
        return imagem_rgb
    if random.random() > probabilidade:
        return imagem_rgb

    copia = imagem_rgb.copy()
    altura, largura = copia.shape[:2]
    fracao_h = random.uniform(fracao_altura_min, fracao_altura_max)
    alt_faixa = max(8, int(round(altura * fracao_h)))

    amostra_castelo = random.choice(banco_castelos)
    angulo_castelo = random.choice((0, 90, 180, 270))
    if angulo_castelo == 90:
        amostra_castelo = cv2.rotate(amostra_castelo, cv2.ROTATE_90_CLOCKWISE)
    elif angulo_castelo == 180:
        amostra_castelo = cv2.rotate(amostra_castelo, cv2.ROTATE_180)
    elif angulo_castelo == 270:
        amostra_castelo = cv2.rotate(amostra_castelo, cv2.ROTATE_90_COUNTERCLOCKWISE)

    alt_c, larg_c = amostra_castelo.shape[:2]
    if larg_c != largura:
        amostra_castelo = cv2.resize(amostra_castelo, (largura, alt_c), interpolation=cv2.INTER_AREA)

    y_ini_castelo = random.randint(0, max(0, amostra_castelo.shape[0] - alt_faixa))
    faixa_castelo = amostra_castelo[y_ini_castelo : y_ini_castelo + alt_faixa, :]

    # Sombra projetada suave na transição entre a torre 3D e o terreno real
    y_topo_oclusao = altura - alt_faixa
    alt_sombra = min(6, y_topo_oclusao)
    if alt_sombra > 0:
        for dy in range(alt_sombra):
            y_s = y_topo_oclusao - alt_sombra + dy
            fator_escurecimento = 0.55 + 0.45 * ((alt_sombra - dy) / float(alt_sombra))
            copia[y_s, :] = np.clip(
                copia[y_s, :].astype(np.float32) * fator_escurecimento, 0, 255
            ).astype(np.uint8)

    copia[y_topo_oclusao:altura, :] = faixa_castelo
    return copia


def aplicar_reflexo_especular_flash(imagem_rgb, probabilidade=0.20):
    """
    Em 20% das amostras de treino, aplica uma mancha elíptica suave de brilho intenso
    para simular o estouro de flash / luz especular sobre o verniz da peça.
    """
    if random.random() > probabilidade:
        return imagem_rgb

    altura, largura = imagem_rgb.shape[:2]
    mascara_brilho = np.zeros((altura, largura), dtype=np.float32)

    centro_x = random.randint(int(largura * 0.20), int(largura * 0.80))
    centro_y = random.randint(int(altura * 0.20), int(altura * 0.80))
    eixo_a = random.randint(int(largura * 0.16), int(largura * 0.38))
    eixo_b = random.randint(int(altura * 0.14), int(altura * 0.34))
    angulo = random.randint(0, 180)

    cv2.ellipse(
        mascara_brilho,
        (centro_x, centro_y),
        (eixo_a, eixo_b),
        angulo,
        0,
        360,
        1.0,
        -1,
    )
    sigma = max(7.0, min(largura, altura) * 0.12)
    mascara_suave = cv2.GaussianBlur(mascara_brilho, (0, 0), sigmaX=sigma, sigmaY=sigma)
    max_val = float(np.max(mascara_suave))
    if max_val > 1e-6:
        mascara_suave = mascara_suave / max_val

    intensidade_pico = random.uniform(0.55, 0.88)
    alfa = (mascara_suave * intensidade_pico)[..., np.newaxis]

    cor_reflexo = np.array([255.0, 253.0, 245.0], dtype=np.float32)
    img_float = imagem_rgb.astype(np.float32)
    combinada = img_float * (1.0 - alfa) + cor_reflexo * alfa
    return np.clip(combinada, 0, 255).astype(np.uint8)


class ConjuntoDadosCelulasFase1(Dataset):
    def __init__(
        self,
        amostras,
        banco_castelos=None,
        tamanho_imagem=160,
        treino=True,
    ):
        self.amostras = amostras
        self.banco_castelos = banco_castelos or []
        self.tamanho_imagem = tamanho_imagem
        self.treino = treino
        self.variacao_luz = transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.08,
            hue=0.0,
        )
        self.para_tensor_normalizado = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=MEDIA_IMAGENET, std=DESVIO_IMAGENET),
        ])

    def __len__(self):
        return len(self.amostras)

    def gerar_imagem_aumentada_rgb(self, item):
        img_bgr = cv2.imread(item["caminho"])
        if img_bgr is None:
            raise FileNotFoundError(f"Não foi possível ler a célula: {item['caminho']}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(
            img_rgb,
            (self.tamanho_imagem, self.tamanho_imagem),
            interpolation=cv2.INTER_AREA,
        )

        if not self.treino:
            return img_rgb

        # 1. Deslocamento aleatório de recorte (+-6 px)
        img_rgb = aplicar_deslocamento_aleatorio_recorte(img_rgb, deslocamento_max_px=6)

        # 2. Simulação de Oclusão por Torre 3D do Castelo na borda inferior (25%)
        img_rgb = aplicar_oclusao_torre_castelo(
            img_rgb,
            terreno=item["terreno"],
            banco_castelos=self.banco_castelos,
            probabilidade=0.25,
            fracao_altura_min=0.20,
            fracao_altura_max=0.35,
        )

        # 3. Simulação de Reflexo Especular / Flash (20%)
        img_rgb = aplicar_reflexo_especular_flash(img_rgb, probabilidade=0.20)

        # 4. Rotações ortogonais aleatórias (0, 90, 180, 270 graus) e flips
        angulo = random.choice((0, 90, 180, 270))
        if angulo == 90:
            img_rgb = cv2.rotate(img_rgb, cv2.ROTATE_90_CLOCKWISE)
        elif angulo == 180:
            img_rgb = cv2.rotate(img_rgb, cv2.ROTATE_180)
        elif angulo == 270:
            img_rgb = cv2.rotate(img_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)

        if random.random() < 0.5:
            img_rgb = cv2.flip(img_rgb, 1)

        # 5. Variação moderada de brilho/contraste (+-15%) com matiz (hue) fixa em 0
        img_pil = Image.fromarray(img_rgb)
        img_pil = self.variacao_luz(img_pil)
        return np.asarray(img_pil, dtype=np.uint8)

    def __getitem__(self, indice):
        item = self.amostras[indice]
        img_rgb = self.gerar_imagem_aumentada_rgb(item)
        img_pil = Image.fromarray(img_rgb)
        tensor_img = self.para_tensor_normalizado(img_pil)

        alvo_terreno = MAPA_TERRENO_IDX[item["terreno"]]
        alvo_coroas = MAPA_COROAS_IDX[int(item["coroas"])]
        return tensor_img, alvo_terreno, alvo_coroas, item["prefixo"], item["arquivo"]

In [ ]:
# Visualização rápida dos Augmentations Físicos Direcionados (Oclusão Torre 3D + Reflexo Flash)
def visualizar_exemplos_augmentation_fisico(amostras_treino, banco_castelos):
    elegiveis = [a for a in amostras_treino if a["terreno"] not in ("castelo", "vazio")]
    selecionadas = random.sample(elegiveis, min(4, len(elegiveis)))

    fig, eixos = plt.subplots(2, 4, figsize=(13, 6.5), dpi=130)
    fig.suptitle(
        "Auditoria Visual do Augmentation Físico (Topo: Original | Base: Oclusão Torre 3D + Reflexo Flash)",
        fontsize=12,
        fontweight="bold",
    )

    for col, item in enumerate(selecionadas):
        img_bgr = cv2.imread(item["caminho"])
        img_orig = cv2.resize(
            cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
            (TAMANHO_IMAGEM, TAMANHO_IMAGEM),
        )
        img_aug = aplicar_deslocamento_aleatorio_recorte(img_orig, deslocamento_max_px=6)
        img_aug = aplicar_oclusao_torre_castelo(
            img_aug,
            terreno=item["terreno"],
            banco_castelos=banco_castelos,
            probabilidade=1.0,
        )
        img_aug = aplicar_reflexo_especular_flash(img_aug, probabilidade=1.0)

        eixos[0, col].imshow(img_orig)
        eixos[0, col].set_title(f"Original: {item['terreno']} ({item['coroas']}c)", fontsize=9.5)
        eixos[0, col].axis("off")

        eixos[1, col].imshow(img_aug)
        eixos[1, col].set_title("Oclusão 3D + Flash Especular", fontsize=9.5, color="#b22222")
        eixos[1, col].axis("off")

    plt.tight_layout()
    plt.show()


visualizar_exemplos_augmentation_fisico(AMOSTRAS_TREINO, BANCO_CASTELOS_TREINO)

## 3. Balanceamento de Classes
Combina duas estratégias complementares para combater o desbalanceamento natural do Kingdomino (onde `0 coroas` representa ~69% da base e `2c`/`3c` ou `mineracao`/`pantano`/`vazio` são minoritários):
1. **`WeightedRandomSampler` no `DataLoader` de treino:** atribui peso a cada célula de treino combinando a raridade do terreno e a raridade da classe de coroas.
2. **`CrossEntropyLoss` com pesos inversos à frequência das classes e `label_smoothing` ajustável:** reforça o gradiente das classes raras nas duas cabeças (`cabeca_terreno` e `cabeca_coroas`).

In [ ]:
def calcular_pesos_inversos_classes(contagens, potencia=0.75, limite_maximo=6.0):
    arr = np.maximum(1.0, np.array(contagens, dtype=np.float32))
    total = float(np.sum(arr))
    num_classes = len(arr)
    pesos = total / (num_classes * arr)
    pesos = np.power(pesos, potencia)
    pesos = pesos / np.mean(pesos)
    pesos = np.clip(pesos, 0.30, limite_maximo)
    return pesos.astype(np.float32)


def construir_pesos_e_amostrador_treino(amostras_treino, dispositivo):
    contagem_terrenos = [0] * len(CLASSES_TERRENO)
    contagem_coroas = [0] * len(CLASSES_COROAS)

    for item in amostras_treino:
        contagem_terrenos[MAPA_TERRENO_IDX[item["terreno"]]] += 1
        contagem_coroas[MAPA_COROAS_IDX[int(item["coroas"])]] += 1

    pesos_loss_terreno_np = calcular_pesos_inversos_classes(
        contagem_terrenos, potencia=0.75, limite_maximo=5.5
    )
    pesos_loss_coroas_np = calcular_pesos_inversos_classes(
        contagem_coroas, potencia=0.80, limite_maximo=6.5
    )

    pesos_amostras = []
    for item in amostras_treino:
        idx_t = MAPA_TERRENO_IDX[item["terreno"]]
        idx_c = MAPA_COROAS_IDX[int(item["coroas"])]
        peso_combinado = 0.50 * float(pesos_loss_terreno_np[idx_t]) + 0.50 * float(
            pesos_loss_coroas_np[idx_c]
        )
        pesos_amostras.append(peso_combinado)

    pesos_amostras_tensor = torch.DoubleTensor(pesos_amostras)
    amostrador = WeightedRandomSampler(
        weights=pesos_amostras_tensor,
        num_samples=len(pesos_amostras_tensor),
        replacement=True,
    )

    tensor_pesos_terreno = torch.tensor(
        pesos_loss_terreno_np, dtype=torch.float32, device=dispositivo
    )
    tensor_pesos_coroas = torch.tensor(
        pesos_loss_coroas_np, dtype=torch.float32, device=dispositivo
    )

    return (
        amostrador,
        tensor_pesos_terreno,
        tensor_pesos_coroas,
        contagem_terrenos,
        contagem_coroas,
    )


(
    AMOSTRADOR_PONDERADO_TREINO,
    PESOS_LOSS_TERRENO,
    PESOS_LOSS_COROAS,
    CONTAGEM_TERRENOS_TREINO,
    CONTAGEM_COROAS_TREINO,
) = construir_pesos_e_amostrador_treino(AMOSTRAS_TREINO, DISPOSITIVO)

print("Pesos de classe para CrossEntropyLoss (Terrenos):")
for idx, nome in enumerate(CLASSES_TERRENO):
    print(f"  - {nome:<12} (n={CONTAGEM_TERRENOS_TREINO[idx]:4d}): peso = {PESOS_LOSS_TERRENO[idx].item():.3f}")

print("\nPesos de classe para CrossEntropyLoss (Coroas):")
for idx, c in enumerate(CLASSES_COROAS):
    print(f"  - {c} coroa(s)   (n={CONTAGEM_COROAS_TREINO[idx]:4d}): peso = {PESOS_LOSS_COROAS[idx].item():.3f}")

## 4. Estudo Automatizado com Optuna (30 Trials)
Explora o espaço de busca de arquiteturas e hiperparâmetros para maximizar a métrica composta na validação:
$$\text{Score Composto} = 0.40 \times \text{F1}_{\text{macro\_terreno}} + 0.45 \times \text{F1}_{\text{macro\_coroas}} + 0.15 \times \text{Taxa}_{\text{Tabuleiros\_100\_Perfeitos}}$$

### Espaço de Busca:
- **Arquitetura:**
  - `mobilenet_v3_small` (inicializado a partir do checkpoint `modelos/classificador_celulas.pth` do Baseline `C0`);
  - `efficientnet_b0` (pesos pré-treinados ImageNet);
  - `convnext_atto` (via `timm`, pesos pré-treinados ImageNet);
  - Restrição rígida: tamanho final em **FP32 `<= 20 MB`**.
- **Taxa de aprendizado (`taxa_aprendizado`):** `5e-5` a `8e-4` (escala log) com `CosineAnnealingLR`;
- **`weight_decay`:** `1e-5` a `1e-2` (escala log);
- **`dropout`:** `0.1` a `0.4`;
- **`label_smoothing`:** `0.0` a `0.08`.

In [ ]:
class ClassificadorMultiTarefaFase1(nn.Module):
    """
    Classificador multi-tarefa unificado compatível com 'mobilenet_v3_small',
    'efficientnet_b0' e 'convnext_atto' (via timm), produzindo
    (logits_terreno, logits_coroas).
    """

    def __init__(
        self,
        arquitetura="mobilenet_v3_small",
        num_terrenos=8,
        num_coroas=4,
        taxa_dropout=0.25,
        pre_treinado=True,
        caminho_checkpoint_c0=None,
    ):
        super().__init__()
        self.arquitetura = arquitetura
        self.usa_timm = False

        if arquitetura == "mobilenet_v3_small":
            pesos = models.MobileNet_V3_Small_Weights.DEFAULT if pre_treinado else None
            base = models.mobilenet_v3_small(weights=pesos)
            self.extrator = base.features
            self.agrupamento = base.avgpool
            dim_caracteristicas = 576
        elif arquitetura == "efficientnet_b0":
            pesos = models.EfficientNet_B0_Weights.DEFAULT if pre_treinado else None
            base = models.efficientnet_b0(weights=pesos)
            self.extrator = base.features
            self.agrupamento = base.avgpool
            dim_caracteristicas = 1280
        elif arquitetura == "convnext_atto":
            self.usa_timm = True
            self.backbone_timm = timm.create_model(
                "convnext_atto",
                pretrained=pre_treinado,
                num_classes=0,
                global_pool="avg",
            )
            dim_caracteristicas = int(self.backbone_timm.num_features)
        else:
            raise ValueError(f"Arquitetura não suportada: {arquitetura}")

        self.normalizacao = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(p=taxa_dropout),
        )
        self.cabeca_terreno = nn.Linear(dim_caracteristicas, num_terrenos)
        self.cabeca_coroas = nn.Linear(dim_caracteristicas, num_coroas)

        if (
            arquitetura == "mobilenet_v3_small"
            and caminho_checkpoint_c0
            and os.path.isfile(caminho_checkpoint_c0)
        ):
            pacote = torch.load(caminho_checkpoint_c0, map_location="cpu", weights_only=False)
            if pacote.get("arquitetura") == "mobilenet_v3_small" and "estado_modelo" in pacote:
                self.load_state_dict(pacote["estado_modelo"], strict=False)

    def forward(self, imagem):
        if self.usa_timm:
            vetor = self.backbone_timm(imagem)
        else:
            mapa = self.extrator(imagem)
            vetor = self.agrupamento(mapa)
        vetor = self.normalizacao(vetor)
        logits_terreno = self.cabeca_terreno(vetor)
        logits_coroas = self.cabeca_coroas(vetor)
        return logits_terreno, logits_coroas


def calcular_tamanho_modelo_fp32_mb(modelo):
    total_bytes = 0
    for param in modelo.parameters():
        total_bytes += param.numel() * 4
    for buf in modelo.buffers():
        total_bytes += buf.numel() * 4
    return total_bytes / (1024.0 * 1024.0)


for arq_teste in ("mobilenet_v3_small", "efficientnet_b0", "convnext_atto"):
    mod_tmp = ClassificadorMultiTarefaFase1(
        arquitetura=arq_teste,
        pre_treinado=False,
        caminho_checkpoint_c0=CAMINHO_CHECKPOINT_C0,
    )
    tam_mb = calcular_tamanho_modelo_fp32_mb(mod_tmp)
    print(f"Arquitetura {arq_teste:<20} -> Tamanho estimado em FP32: {tam_mb:.2f} MB (Limite <= {LIMITE_TAMANHO_FP32_MB:.1f} MB)")
    del mod_tmp

In [ ]:
def avaliar_modelo_completo(modelo, carregador_val, dispositivo):
    modelo.eval()
    y_real_t = []
    y_pred_t = []
    y_real_c = []
    y_pred_c = []
    erros_por_tabuleiro = {}
    total_por_tabuleiro = {}

    idx_castelo = MAPA_TERRENO_IDX["castelo"]
    idx_vazio = MAPA_TERRENO_IDX["vazio"]

    with torch.no_grad():
        for lote_imgs, alvos_t, alvos_c, prefixos, _ in carregador_val:
            lote_imgs = lote_imgs.to(dispositivo, non_blocking=True)
            with torch.amp.autocast(TIPO_AMP, enabled=USAR_AMP):
                logits_t, logits_c = modelo(lote_imgs)

            preds_t = torch.argmax(logits_t, dim=1).cpu().numpy()
            preds_c = torch.argmax(logits_c, dim=1).cpu().numpy()
            reais_t = alvos_t.numpy()
            reais_c = alvos_c.numpy()

            for i in range(len(reais_t)):
                pt = int(preds_t[i])
                pc = int(preds_c[i])
                if pt in (idx_castelo, idx_vazio):
                    pc = 0

                rt = int(reais_t[i])
                rc = int(reais_c[i])
                pref = prefixos[i]

                y_real_t.append(rt)
                y_pred_t.append(pt)
                y_real_c.append(rc)
                y_pred_c.append(pc)

                total_por_tabuleiro[pref] = total_por_tabuleiro.get(pref, 0) + 1
                if pref not in erros_por_tabuleiro:
                    erros_por_tabuleiro[pref] = 0
                if pt != rt or pc != rc:
                    erros_por_tabuleiro[pref] += 1

    acc_terreno = float(accuracy_score(y_real_t, y_pred_t))
    f1_macro_terreno = float(
        f1_score(
            y_real_t,
            y_pred_t,
            labels=list(range(len(CLASSES_TERRENO))),
            average="macro",
            zero_division=0,
        )
    )
    f1_por_classe_t = f1_score(
        y_real_t,
        y_pred_t,
        labels=list(range(len(CLASSES_TERRENO))),
        average=None,
        zero_division=0,
    )

    acc_coroas = float(accuracy_score(y_real_c, y_pred_c))
    f1_macro_coroas = float(
        f1_score(
            y_real_c,
            y_pred_c,
            labels=list(range(len(CLASSES_COROAS))),
            average="macro",
            zero_division=0,
        )
    )
    f1_por_classe_c = f1_score(
        y_real_c,
        y_pred_c,
        labels=list(range(len(CLASSES_COROAS))),
        average=None,
        zero_division=0,
    )

    acertos_exatos = sum(
        1 for rt, pt, rc, pc in zip(y_real_t, y_pred_t, y_real_c, y_pred_c)
        if rt == pt and rc == pc
    )
    acc_celula_exata = acertos_exatos / float(max(1, len(y_real_t)))

    total_tabuleiros = len(erros_por_tabuleiro)
    tabuleiros_perfeitos = sum(1 for qtd in erros_por_tabuleiro.values() if qtd == 0)
    taxa_tabuleiros_perfeitos = (
        tabuleiros_perfeitos / float(total_tabuleiros) if total_tabuleiros > 0 else 0.0
    )

    score_composto = (
        0.40 * f1_macro_terreno
        + 0.45 * f1_macro_coroas
        + 0.15 * taxa_tabuleiros_perfeitos
    )

    mat_conf_terreno = confusion_matrix(
        y_real_t, y_pred_t, labels=list(range(len(CLASSES_TERRENO)))
    ).tolist()
    mat_conf_coroas = confusion_matrix(
        y_real_c, y_pred_c, labels=list(range(len(CLASSES_COROAS)))
    ).tolist()

    return {
        "score_composto": float(score_composto),
        "acuracia_terreno": float(acc_terreno),
        "acuracia_terreno_pct": round(acc_terreno * 100.0, 2),
        "f1_macro_terreno": float(f1_macro_terreno),
        "f1_por_classe_terreno": {
            CLASSES_TERRENO[i]: round(float(f1_por_classe_t[i]), 4)
            for i in range(len(CLASSES_TERRENO))
        },
        "acuracia_coroas": float(acc_coroas),
        "acuracia_coroas_pct": round(acc_coroas * 100.0, 2),
        "f1_macro_coroas": float(f1_macro_coroas),
        "f1_por_classe_coroas": {
            str(CLASSES_COROAS[i]): round(float(f1_por_classe_c[i]), 4)
            for i in range(len(CLASSES_COROAS))
        },
        "acuracia_celula_exata": float(acc_celula_exata),
        "acuracia_celula_exata_pct": round(acc_celula_exata * 100.0, 2),
        "total_celulas_avaliadas": len(y_real_t),
        "total_celulas_com_erro": len(y_real_t) - acertos_exatos,
        "tabuleiros_100_perfeitos": int(tabuleiros_perfeitos),
        "total_tabuleiros_avaliados": int(total_tabuleiros),
        "taxa_tabuleiros_perfeitos": round(taxa_tabuleiros_perfeitos * 100.0, 2),
        "taxa_tabuleiros_perfeitos_fracao": float(taxa_tabuleiros_perfeitos),
        "matriz_confusao_terreno": mat_conf_terreno,
        "matriz_confusao_coroas": mat_conf_coroas,
    }

In [ ]:
# Preparação dos DataLoaders de Treino e Validação
TAMANHO_LOTE = 64 if TEM_GPU else 32
NUM_TRABALHADORES = 2 if TEM_GPU else 0

CONJUNTO_TREINO = ConjuntoDadosCelulasFase1(
    amostras=AMOSTRAS_TREINO,
    banco_castelos=BANCO_CASTELOS_TREINO,
    tamanho_imagem=TAMANHO_IMAGEM,
    treino=True,
)
CONJUNTO_VALIDACAO = ConjuntoDadosCelulasFase1(
    amostras=AMOSTRAS_VALIDACAO,
    banco_castelos=None,
    tamanho_imagem=TAMANHO_IMAGEM,
    treino=False,
)

CARREGADOR_TREINO = DataLoader(
    CONJUNTO_TREINO,
    batch_size=TAMANHO_LOTE,
    sampler=AMOSTRADOR_PONDERADO_TREINO,
    num_workers=NUM_TRABALHADORES,
    pin_memory=TEM_GPU,
)
CARREGADOR_VALIDACAO = DataLoader(
    CONJUNTO_VALIDACAO,
    batch_size=TAMANHO_LOTE,
    shuffle=False,
    num_workers=NUM_TRABALHADORES,
    pin_memory=TEM_GPU,
)

print(f"Lotes por época (Treino): {len(CARREGADOR_TREINO)} | Lotes (Validação): {len(CARREGADOR_VALIDACAO)}")

In [ ]:
N_TRIALS_OPTUNA = 30
EPOCAS_POR_TRIAL = 18
CAMINHO_MELHOR_PTH = os.path.join(PASTA_RESULTADOS_FASE1, "melhor_classificador_fase1.pth")

HISTORICO_TRIALS = []
ESTADO_MELHOR_GLOBAL = {
    "score_composto": -1.0,
    "numero_trial": -1,
    "parametros": None,
    "metricas_validacao": None,
    "tamanho_fp32_mb": 0.0,
}


def objetivo_optuna(trial):
    fixar_sementes(SEMENTE_GLOBAL + trial.number)

    arquitetura = trial.suggest_categorical(
        "arquitetura",
        ["mobilenet_v3_small", "efficientnet_b0", "convnext_atto"],
    )
    taxa_aprendizado = trial.suggest_float("taxa_aprendizado", 5e-5, 8e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.10, 0.40)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.08)

    modelo = ClassificadorMultiTarefaFase1(
        arquitetura=arquitetura,
        num_terrenos=len(CLASSES_TERRENO),
        num_coroas=len(CLASSES_COROAS),
        taxa_dropout=dropout,
        pre_treinado=True,
        caminho_checkpoint_c0=CAMINHO_CHECKPOINT_C0,
    )

    tamanho_mb = calcular_tamanho_modelo_fp32_mb(modelo)
    trial.set_user_attr("tamanho_fp32_mb", round(tamanho_mb, 3))

    if tamanho_mb > LIMITE_TAMANHO_FP32_MB:
        raise optuna.TrialPruned(
            f"Modelo {arquitetura} excedeu o limite de {LIMITE_TAMANHO_FP32_MB} MB ({tamanho_mb:.2f} MB)."
        )

    modelo.to(DISPOSITIVO)

    criterio_terreno = nn.CrossEntropyLoss(
        weight=PESOS_LOSS_TERRENO,
        label_smoothing=label_smoothing,
    )
    criterio_coroas = nn.CrossEntropyLoss(
        weight=PESOS_LOSS_COROAS,
        label_smoothing=label_smoothing,
    )

    otimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=taxa_aprendizado,
        weight_decay=weight_decay,
    )
    agendador = torch.optim.lr_scheduler.CosineAnnealingLR(
        otimizador,
        T_max=EPOCAS_POR_TRIAL,
        eta_min=taxa_aprendizado * 0.05,
    )
    escalador_amp = torch.amp.GradScaler(TIPO_AMP, enabled=USAR_AMP)

    melhor_score_trial = -1.0
    melhores_metricas_trial = None
    melhor_estado_pesos = None

    for epoca in range(1, EPOCAS_POR_TRIAL + 1):
        modelo.train()
        perda_acumulada = 0.0
        total_lotes = 0

        for lote_imgs, alvos_t, alvos_c, _, _ in CARREGADOR_TREINO:
            lote_imgs = lote_imgs.to(DISPOSITIVO, non_blocking=True)
            alvos_t = alvos_t.to(DISPOSITIVO, non_blocking=True)
            alvos_c = alvos_c.to(DISPOSITIVO, non_blocking=True)

            otimizador.zero_grad(set_to_none=True)
            with torch.amp.autocast(TIPO_AMP, enabled=USAR_AMP):
                logits_t, logits_c = modelo(lote_imgs)
                perda_t = criterio_terreno(logits_t, alvos_t)
                perda_c = criterio_coroas(logits_c, alvos_c)
                perda_total = 1.0 * perda_t + 1.15 * perda_c

            escalador_amp.scale(perda_total).backward()
            escalador_amp.step(otimizador)
            escalador_amp.update()

            perda_acumulada += float(perda_total.item())
            total_lotes += 1

        agendador.step()

        metricas_val = avaliar_modelo_completo(modelo, CARREGADOR_VALIDACAO, DISPOSITIVO)
        score_atual = metricas_val["score_composto"]

        if score_atual > melhor_score_trial:
            melhor_score_trial = score_atual
            melhores_metricas_trial = metricas_val
            melhor_estado_pesos = copy.deepcopy(modelo.state_dict())

        trial.report(score_atual, step=epoca)
        if trial.should_prune() and epoca >= 6:
            raise optuna.TrialPruned()

    trial.set_user_attr("f1_macro_terreno", round(melhores_metricas_trial["f1_macro_terreno"], 4))
    trial.set_user_attr("f1_macro_coroas", round(melhores_metricas_trial["f1_macro_coroas"], 4))
    trial.set_user_attr(
        "taxa_tabuleiros_perfeitos",
        round(melhores_metricas_trial["taxa_tabuleiros_perfeitos"], 2),
    )
    trial.set_user_attr(
        "acuracia_celula_exata_pct",
        round(melhores_metricas_trial["acuracia_celula_exata_pct"], 2),
    )

    registro_historico = {
        "trial": int(trial.number),
        "arquitetura": arquitetura,
        "taxa_aprendizado": float(taxa_aprendizado),
        "weight_decay": float(weight_decay),
        "dropout": float(dropout),
        "label_smoothing": float(label_smoothing),
        "tamanho_fp32_mb": round(tamanho_mb, 3),
        "score_composto": round(melhor_score_trial, 4),
        "f1_macro_terreno": round(melhores_metricas_trial["f1_macro_terreno"], 4),
        "f1_macro_coroas": round(melhores_metricas_trial["f1_macro_coroas"], 4),
        "acuracia_celula_exata_pct": round(melhores_metricas_trial["acuracia_celula_exata_pct"], 2),
        "taxa_tabuleiros_perfeitos": round(melhores_metricas_trial["taxa_tabuleiros_perfeitos"], 2),
    }
    HISTORICO_TRIALS.append(registro_historico)

    if melhor_score_trial > ESTADO_MELHOR_GLOBAL["score_composto"]:
        ESTADO_MELHOR_GLOBAL["score_composto"] = melhor_score_trial
        ESTADO_MELHOR_GLOBAL["numero_trial"] = int(trial.number)
        ESTADO_MELHOR_GLOBAL["parametros"] = dict(trial.params)
        ESTADO_MELHOR_GLOBAL["metricas_validacao"] = melhores_metricas_trial
        ESTADO_MELHOR_GLOBAL["tamanho_fp32_mb"] = round(tamanho_mb, 3)

        pacote_salvar = {
            "arquitetura": arquitetura,
            "estado_modelo": melhor_estado_pesos,
            "classes_terreno": CLASSES_TERRENO,
            "classes_coroas": CLASSES_COROAS,
            "tamanho_imagem": TAMANHO_IMAGEM,
            "dropout": dropout,
            "hiperparametros": dict(trial.params),
            "metricas_validacao": melhores_metricas_trial,
            "score_composto": melhor_score_trial,
            "tamanho_fp32_mb": round(tamanho_mb, 3),
        }
        torch.save(pacote_salvar, CAMINHO_MELHOR_PTH)
        print(
            f"\n[Trial {trial.number:02d}] NOVO MELHOR MODELO! "
            f"Arq={arquitetura} | Score={melhor_score_trial:.4f} "
            f"(F1-T={melhores_metricas_trial['f1_macro_terreno']:.4f}, "
            f"F1-C={melhores_metricas_trial['f1_macro_coroas']:.4f}, "
            f"TabPerfeitos={melhores_metricas_trial['taxa_tabuleiros_perfeitos']:.2f}%)"
        )

    return melhor_score_trial

In [ ]:
inicio_estudo = time.time()

amostrador_optuna = optuna.samplers.TPESampler(seed=SEMENTE_GLOBAL, multivariate=True)
podador_optuna = optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=6)

ESTUDO_OPTUNA = optuna.create_study(
    study_name="fase1_classificador_kingdomino",
    direction="maximize",
    sampler=amostrador_optuna,
    pruner=podador_optuna,
)

# Enfileira como primeiro trial o ponto de partida fiel ao Baseline C0 (mobilenet_v3_small)
ESTUDO_OPTUNA.enqueue_trial({
    "arquitetura": "mobilenet_v3_small",
    "taxa_aprendizado": 2.5e-4,
    "weight_decay": 1e-4,
    "dropout": 0.25,
    "label_smoothing": 0.03,
})

ESTUDO_OPTUNA.optimize(objetivo_optuna, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

tempo_estudo_min = (time.time() - inicio_estudo) / 60.0
print("\n" + "=" * 78)
print(f"ESTUDO OPTUNA CONCLUÍDO EM {tempo_estudo_min:.2f} MINUTOS")
print(f"Melhor Trial: #{ESTUDO_OPTUNA.best_trial.number}")
print(f"Melhor Score Composto: {ESTUDO_OPTUNA.best_value:.4f} (Baseline C0: {SCORE_COMPOSTO_BASELINE_C0:.4f})")
print("Melhores Hiperparâmetros:")
for chave, valor in ESTUDO_OPTUNA.best_params.items():
    print(f"  - {chave}: {valor}")
print("=" * 78)

## 5. Exportação ONNX em FP32 e Gráficos Coloridos de Auditoria (`resultados_fase1/`)
- Exporta o melhor modelo do estudo em precisão total **FP32** (`resultados_fase1/melhor_classificador_fase1_fp32.onnx`) com entrada dinâmica `"imagem"` `[lote, 3, 160, 160]` e saídas `["logits_terreno", "logits_coroas"]`, validando a paridade numérica no `onnxruntime`.
- Gera, exibe inline e salva em alta resolução (`300 dpi`) os 4 gráficos de auditoria:
  1. `comparativo_c0_vs_melhor.png`
  2. `matrizes_confusao_fase1.png`
  3. `importancia_hiperparametros_optuna.png`
  4. `fronteira_pareto_classificador.png`
- Salva o relatório consolidado em `resultados_fase1/relatorio_fase1.json`.

In [ ]:
CAMINHO_ONNX_FASE1 = os.path.join(
    PASTA_RESULTADOS_FASE1, "melhor_classificador_fase1_fp32.onnx"
)


def exportar_e_validar_melhor_modelo_onnx(caminho_pth, caminho_onnx):
    pacote = torch.load(caminho_pth, map_location="cpu", weights_only=False)
    arquitetura = pacote["arquitetura"]
    dropout = float(pacote.get("dropout", 0.25))

    modelo_cpu = ClassificadorMultiTarefaFase1(
        arquitetura=arquitetura,
        num_terrenos=len(CLASSES_TERRENO),
        num_coroas=len(CLASSES_COROAS),
        taxa_dropout=dropout,
        pre_treinado=False,
    )
    modelo_cpu.load_state_dict(pacote["estado_modelo"])
    modelo_cpu.eval()

    entrada_exemplo = torch.randn(4, 3, TAMANHO_IMAGEM, TAMANHO_IMAGEM, dtype=torch.float32)
    with torch.no_grad():
        saida_pt_t, saida_pt_c = modelo_cpu(entrada_exemplo)
        torch.onnx.export(
            modelo_cpu,
            entrada_exemplo,
            caminho_onnx,
            export_params=True,
            opset_version=17,
            do_constant_folding=True,
            input_names=["imagem"],
            output_names=["logits_terreno", "logits_coroas"],
            dynamic_axes={
                "imagem": {0: "lote"},
                "logits_terreno": {0: "lote"},
                "logits_coroas": {0: "lote"},
            },
            dynamo=False,
        )

    grafo_onnx = onnx.load(caminho_onnx)
    onnx.checker.check_model(grafo_onnx)

    sessao_ort = ort.InferenceSession(caminho_onnx, providers=["CPUExecutionProvider"])
    saida_ort_t, saida_ort_c = sessao_ort.run(
        ["logits_terreno", "logits_coroas"],
        {"imagem": entrada_exemplo.numpy()},
    )

    erro_max_t = float(np.max(np.abs(saida_pt_t.numpy() - saida_ort_t)))
    erro_max_c = float(np.max(np.abs(saida_pt_c.numpy() - saida_ort_c)))
    tamanho_arquivo_mb = os.path.getsize(caminho_onnx) / (1024.0 * 1024.0)

    print("EXPORTAÇÃO E VALIDAÇÃO ONNX FP32 CONCLUÍDA:")
    print(f"  - Arquivo ONNX:           {caminho_onnx}")
    print(f"  - Arquitetura:            {arquitetura}")
    print(f"  - Tamanho em disco:       {tamanho_arquivo_mb:.2f} MB (FP32)")
    print(f"  - Diferença máx PyTorch vs ONNX (Terreno): {erro_max_t:.6e}")
    print(f"  - Diferença máx PyTorch vs ONNX (Coroas):  {erro_max_c:.6e}")

    return {
        "caminho_onnx": caminho_onnx,
        "arquitetura": arquitetura,
        "precisao": "FP32",
        "tamanho_onnx_mb": round(tamanho_arquivo_mb, 3),
        "erro_max_paridade_terreno": erro_max_t,
        "erro_max_paridade_coroas": erro_max_c,
    }


INFO_EXPORTACAO_ONNX = exportar_e_validar_melhor_modelo_onnx(
    CAMINHO_MELHOR_PTH, CAMINHO_ONNX_FASE1
)

In [ ]:
# 5.1 Gráfico 1: comparativo_c0_vs_melhor.png (Barras Agrupadas)
def plotar_comparativo_c0_vs_melhor(baseline_c0, melhor_val, caminho_saida):
    metricas_nomes = [
        "F1-Macro Terreno\n(8 classes)",
        "F1-Macro Coroas\n(4 classes)",
        "Acurácia Célula\nExata (%)",
        "Tabuleiros 100%\nPerfeitos (%)",
    ]
    valores_c0 = [
        float(baseline_c0["f1_macro_terreno"]) * 100.0,
        float(baseline_c0["f1_macro_coroas"]) * 100.0,
        float(baseline_c0["acuracia_celula_exata_pct"]),
        float(baseline_c0["taxa_tabuleiros_perfeitos"]),
    ]
    valores_novo = [
        float(melhor_val["f1_macro_terreno"]) * 100.0,
        float(melhor_val["f1_macro_coroas"]) * 100.0,
        float(melhor_val["acuracia_celula_exata_pct"]),
        float(melhor_val["taxa_tabuleiros_perfeitos"]),
    ]

    x = np.arange(len(metricas_nomes))
    largura = 0.35

    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(10.5, 6.2), dpi=300)

    barras_c0 = ax.bar(
        x - largura / 2,
        valores_c0,
        largura,
        label="Baseline C0 (MobileNetV3-Small Original)",
        color="#5c6bc0",
        edgecolor="#283593",
        linewidth=1.1,
    )
    barras_novo = ax.bar(
        x + largura / 2,
        valores_novo,
        largura,
        label=f"Melhor Fase 1 ({ESTADO_MELHOR_GLOBAL['parametros']['arquitetura']})",
        color="#26a69a",
        edgecolor="#00695c",
        linewidth=1.1,
    )

    ax.set_ylabel("Pontuação / Percentual (%)", fontsize=11, fontweight="bold")
    ax.set_title(
        "Comparativo de Desempenho na Validação: Baseline C0 vs. Melhor Modelo (Fase 1)",
        fontsize=13,
        fontweight="bold",
        pad=14,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(metricas_nomes, fontsize=10.5, fontweight="bold")
    ax.set_ylim(0, 112)
    ax.legend(frameon=True, fontsize=10.5, loc="lower right")

    for barra in barras_c0:
        altura_b = barra.get_height()
        ax.annotate(
            f"{altura_b:.2f}%",
            xy=(barra.get_x() + barra.get_width() / 2, altura_b),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9.5,
            color="#283593",
            fontweight="bold",
        )

    for barra, v_ant in zip(barras_novo, valores_c0):
        altura_b = barra.get_height()
        ganho = altura_b - v_ant
        sinal = "+" if ganho >= 0 else ""
        ax.annotate(
            f"{altura_b:.2f}%\n({sinal}{ganho:.2f} pp)",
            xy=(barra.get_x() + barra.get_width() / 2, altura_b),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            color="#00695c",
            fontweight="bold",
        )

    plt.tight_layout()
    fig.savefig(caminho_saida, dpi=300)
    plt.show()


CAMINHO_GRAFICO_COMPARATIVO = os.path.join(
    PASTA_RESULTADOS_FASE1, "comparativo_c0_vs_melhor.png"
)
plotar_comparativo_c0_vs_melhor(
    BASELINE_VAL_C0,
    ESTADO_MELHOR_GLOBAL["metricas_validacao"],
    CAMINHO_GRAFICO_COMPARATIVO,
)

In [ ]:
# 5.2 Gráfico 2: matrizes_confusao_fase1.png (Heatmaps Lado a Lado: Baseline vs Novo)
def plotar_matrizes_confusao_lado_a_lado(baseline_c0, melhor_val, caminho_saida):
    mat_t_c0 = np.array(baseline_c0["matriz_confusao_terreno"]["matriz"])
    mat_t_novo = np.array(melhor_val["matriz_confusao_terreno"])

    mat_c_c0 = np.array(baseline_c0["matriz_confusao_coroas"]["matriz"])
    mat_c_novo = np.array(melhor_val["matriz_confusao_coroas"])

    fig, eixos = plt.subplots(2, 2, figsize=(15, 12.5), dpi=300)
    fig.suptitle(
        "Matrizes de Confusão na Validação — Baseline C0 (Esquerda) vs. Melhor Fase 1 (Direita)",
        fontsize=14,
        fontweight="bold",
        y=0.98,
    )

    sns.heatmap(
        mat_t_c0,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASSES_TERRENO,
        yticklabels=CLASSES_TERRENO,
        ax=eixos[0, 0],
        cbar=False,
    )
    eixos[0, 0].set_title(
        f"Terrenos — Baseline C0 (F1-Macro: {baseline_c0['f1_macro_terreno']:.4f})",
        fontsize=11.5,
        fontweight="bold",
    )
    eixos[0, 0].set_ylabel("Classe Real")
    eixos[0, 0].set_xlabel("Classe Predita")
    eixos[0, 0].tick_params(axis="x", rotation=30)

    sns.heatmap(
        mat_t_novo,
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=CLASSES_TERRENO,
        yticklabels=CLASSES_TERRENO,
        ax=eixos[0, 1],
        cbar=False,
    )
    eixos[0, 1].set_title(
        f"Terrenos — Melhor Fase 1 (F1-Macro: {melhor_val['f1_macro_terreno']:.4f})",
        fontsize=11.5,
        fontweight="bold",
    )
    eixos[0, 1].set_ylabel("Classe Real")
    eixos[0, 1].set_xlabel("Classe Predita")
    eixos[0, 1].tick_params(axis="x", rotation=30)

    rotulos_coroas = [f"{c} coroa(s)" for c in CLASSES_COROAS]
    sns.heatmap(
        mat_c_c0,
        annot=True,
        fmt="d",
        cmap="Oranges",
        xticklabels=rotulos_coroas,
        yticklabels=rotulos_coroas,
        ax=eixos[1, 0],
        cbar=False,
    )
    eixos[1, 0].set_title(
        f"Coroas — Baseline C0 (F1-Macro: {baseline_c0['f1_macro_coroas']:.4f})",
        fontsize=11.5,
        fontweight="bold",
    )
    eixos[1, 0].set_ylabel("Classe Real")
    eixos[1, 0].set_xlabel("Classe Predita")

    sns.heatmap(
        mat_c_novo,
        annot=True,
        fmt="d",
        cmap="Teal" if "Teal" in plt.colormaps() else "YlGnBu",
        xticklabels=rotulos_coroas,
        yticklabels=rotulos_coroas,
        ax=eixos[1, 1],
        cbar=False,
    )
    eixos[1, 1].set_title(
        f"Coroas — Melhor Fase 1 (F1-Macro: {melhor_val['f1_macro_coroas']:.4f})",
        fontsize=11.5,
        fontweight="bold",
    )
    eixos[1, 1].set_ylabel("Classe Real")
    eixos[1, 1].set_xlabel("Classe Predita")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(caminho_saida, dpi=300)
    plt.show()


CAMINHO_GRAFICO_MATRIZES = os.path.join(
    PASTA_RESULTADOS_FASE1, "matrizes_confusao_fase1.png"
)
plotar_matrizes_confusao_lado_a_lado(
    BASELINE_VAL_C0,
    ESTADO_MELHOR_GLOBAL["metricas_validacao"],
    CAMINHO_GRAFICO_MATRIZES,
)

In [ ]:
# 5.3 Gráfico 3: importancia_hiperparametros_optuna.png (Impacto Horizontal Colorido)
def plotar_importancia_hiperparametros(estudo, caminho_saida):
    try:
        importancias = optuna.importance.get_param_importances(estudo)
    except Exception:
        # Fallback por variância explicada caso haja poucos trials completos
        nomes_params = ["arquitetura", "taxa_aprendizado", "weight_decay", "dropout", "label_smoothing"]
        importancias = {nome: 1.0 / len(nomes_params) for nome in nomes_params}

    nomes_legiveis = {
        "arquitetura": "Arquitetura (Backbone)",
        "taxa_aprendizado": "Taxa de Aprendizado (LR)",
        "weight_decay": "Weight Decay (L2)",
        "dropout": "Taxa de Dropout",
        "label_smoothing": "Label Smoothing",
    }

    itens_ordenados = sorted(importancias.items(), key=lambda x: x[1], reverse=False)
    rotulos = [nomes_legiveis.get(k, k) for k, _ in itens_ordenados]
    valores = [float(v) * 100.0 for _, v in itens_ordenados]

    paleta = sns.color_palette("viridis", len(rotulos))
    fig, ax = plt.subplots(figsize=(9.5, 5.2), dpi=300)

    barras = ax.barh(rotulos, valores, color=paleta, edgecolor="#263238", height=0.58)
    ax.set_xlabel("Importância Relativa no Score Composto (%)", fontsize=11, fontweight="bold")
    ax.set_title(
        "Importância dos Hiperparâmetros no Estudo Optuna (Fase 1)",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlim(0, max(valores + [25.0]) * 1.18)

    for barra in barras:
        larg_b = barra.get_width()
        ax.annotate(
            f"{larg_b:.1f}%",
            xy=(larg_b, barra.get_y() + barra.get_height() / 2),
            xytext=(6, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=10,
            fontweight="bold",
        )

    plt.tight_layout()
    fig.savefig(caminho_saida, dpi=300)
    plt.show()
    return {k: round(float(v), 4) for k, v in importancias.items()}


CAMINHO_GRAFICO_IMPORTANCIA = os.path.join(
    PASTA_RESULTADOS_FASE1, "importancia_hiperparametros_optuna.png"
)
IMPORTANCIA_HIPERPARAMETROS = plotar_importancia_hiperparametros(
    ESTUDO_OPTUNA, CAMINHO_GRAFICO_IMPORTANCIA
)

In [ ]:
# 5.4 Gráfico 4: fronteira_pareto_classificador.png (Dispersão Eficácia vs Tamanho em MB)
def plotar_fronteira_pareto_trials(historico_trials, score_c0, caminho_saida):
    fig, ax = plt.subplots(figsize=(10.2, 6.2), dpi=300)

    cores_arq = {
        "mobilenet_v3_small": "#1e88e5",
        "efficientnet_b0": "#43a047",
        "convnext_atto": "#fb8c00",
    }

    for arq, cor in cores_arq.items():
        sub = [t for t in historico_trials if t["arquitetura"] == arq]
        if not sub:
            continue
        tamanhos = [t["tamanho_fp32_mb"] for t in sub]
        scores = [t["score_composto"] * 100.0 for t in sub]
        ax.scatter(
            tamanhos,
            scores,
            label=f"{arq} ({len(sub)} trials)",
            color=cor,
            s=85,
            alpha=0.82,
            edgecolors="black",
            linewidth=0.8,
        )

    # Calcula e destaca a Fronteira de Pareto (menor tamanho MB e maior Score Composto)
    ordenados = sorted(historico_trials, key=lambda t: (t["tamanho_fp32_mb"], -t["score_composto"]))
    fronteira = []
    melhor_ate_agora = -1.0
    for item in ordenados:
        if item["score_composto"] > melhor_ate_agora:
            fronteira.append(item)
            melhor_ate_agora = item["score_composto"]

    if fronteira:
        xs_p = [t["tamanho_fp32_mb"] for t in fronteira]
        ys_p = [t["score_composto"] * 100.0 for t in fronteira]
        ax.plot(
            xs_p,
            ys_p,
            linestyle="--",
            color="#d32f2f",
            linewidth=2.0,
            label="Fronteira de Pareto (Eficácia vs. Tamanho)",
        )

    ax.axhline(
        score_c0 * 100.0,
        color="#5e35b1",
        linestyle=":",
        linewidth=1.8,
        label=f"Baseline C0 ({score_c0 * 100.0:.2f}%)",
    )
    ax.axvline(
        LIMITE_TAMANHO_FP32_MB,
        color="#b71c1c",
        linestyle="-.",
        linewidth=1.6,
        label=f"Teto Mobile FP32 ({LIMITE_TAMANHO_FP32_MB:.0f} MB)",
    )

    ax.set_xlabel("Tamanho do Modelo em Precisão Total FP32 (MB)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Score Composto na Validação (%)", fontsize=11, fontweight="bold")
    ax.set_title(
        "Fronteira de Pareto: Eficácia na Validação vs. Tamanho FP32 (MB)",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlim(0, LIMITE_TAMANHO_FP32_MB + 2.5)
    ax.legend(frameon=True, fontsize=9.8, loc="lower right")

    plt.tight_layout()
    fig.savefig(caminho_saida, dpi=300)
    plt.show()


CAMINHO_GRAFICO_PARETO = os.path.join(
    PASTA_RESULTADOS_FASE1, "fronteira_pareto_classificador.png"
)
plotar_fronteira_pareto_trials(
    HISTORICO_TRIALS, SCORE_COMPOSTO_BASELINE_C0, CAMINHO_GRAFICO_PARETO
)

In [ ]:
# 5.5 Consolidação e Salvamento de resultados_fase1/relatorio_fase1.json
CAMINHO_RELATORIO_FASE1 = os.path.join(PASTA_RESULTADOS_FASE1, "relatorio_fase1.json")

melhor_val = ESTADO_MELHOR_GLOBAL["metricas_validacao"]
relatorio_fase1 = {
    "fase": "Fase 1 - Fine-Tuning e Otimizacao Optuna do Classificador Multi-Tarefa",
    "semente": SEMENTE_GLOBAL,
    "dispositivo_execucao": str(DISPOSITIVO),
    "total_trials_executados": len(ESTUDO_OPTUNA.trials),
    "total_trials_concluidos": len(HISTORICO_TRIALS),
    "formula_objetivo": "0.40 * F1_macro_terreno + 0.45 * F1_macro_coroas + 0.15 * Taxa_Tabuleiros_100_Perfeitos",
    "comparativo_validacao": {
        "baseline_c0": {
            "score_composto": round(SCORE_COMPOSTO_BASELINE_C0, 4),
            "acuracia_terreno_pct": BASELINE_VAL_C0["acuracia_terreno_pct"],
            "f1_macro_terreno": BASELINE_VAL_C0["f1_macro_terreno"],
            "acuracia_coroas_pct": BASELINE_VAL_C0["acuracia_coroas_pct"],
            "f1_macro_coroas": BASELINE_VAL_C0["f1_macro_coroas"],
            "acuracia_celula_exata_pct": BASELINE_VAL_C0["acuracia_celula_exata_pct"],
            "taxa_tabuleiros_perfeitos": BASELINE_VAL_C0["taxa_tabuleiros_perfeitos"],
        },
        "melhor_modelo_fase1": {
            "trial_vencedor": ESTADO_MELHOR_GLOBAL["numero_trial"],
            "hiperparametros": ESTADO_MELHOR_GLOBAL["parametros"],
            "tamanho_fp32_mb": ESTADO_MELHOR_GLOBAL["tamanho_fp32_mb"],
            "score_composto": round(melhor_val["score_composto"], 4),
            "acuracia_terreno_pct": melhor_val["acuracia_terreno_pct"],
            "f1_macro_terreno": round(melhor_val["f1_macro_terreno"], 4),
            "f1_por_classe_terreno": melhor_val["f1_por_classe_terreno"],
            "acuracia_coroas_pct": melhor_val["acuracia_coroas_pct"],
            "f1_macro_coroas": round(melhor_val["f1_macro_coroas"], 4),
            "f1_por_classe_coroas": melhor_val["f1_por_classe_coroas"],
            "acuracia_celula_exata_pct": melhor_val["acuracia_celula_exata_pct"],
            "tabuleiros_100_perfeitos": melhor_val["tabuleiros_100_perfeitos"],
            "total_tabuleiros_avaliados": melhor_val["total_tabuleiros_avaliados"],
            "taxa_tabuleiros_perfeitos": melhor_val["taxa_tabuleiros_perfeitos"],
            "matriz_confusao_terreno": melhor_val["matriz_confusao_terreno"],
            "matriz_confusao_coroas": melhor_val["matriz_confusao_coroas"],
        },
        "ganhos_absolutos": {
            "delta_score_composto": round(
                melhor_val["score_composto"] - SCORE_COMPOSTO_BASELINE_C0, 4
            ),
            "delta_f1_macro_terreno": round(
                melhor_val["f1_macro_terreno"] - float(BASELINE_VAL_C0["f1_macro_terreno"]), 4
            ),
            "delta_f1_macro_coroas": round(
                melhor_val["f1_macro_coroas"] - float(BASELINE_VAL_C0["f1_macro_coroas"]), 4
            ),
            "delta_acuracia_celula_exata_pp": round(
                melhor_val["acuracia_celula_exata_pct"]
                - float(BASELINE_VAL_C0["acuracia_celula_exata_pct"]),
                2,
            ),
            "delta_taxa_tabuleiros_perfeitos_pp": round(
                melhor_val["taxa_tabuleiros_perfeitos"]
                - float(BASELINE_VAL_C0["taxa_tabuleiros_perfeitos"]),
                2,
            ),
        },
    },
    "importancia_hiperparametros": IMPORTANCIA_HIPERPARAMETROS,
    "exportacao_onnx_fp32": INFO_EXPORTACAO_ONNX,
    "arquivos_gerados": {
        "checkpoint_pytorch": CAMINHO_MELHOR_PTH,
        "modelo_onnx_fp32": CAMINHO_ONNX_FASE1,
        "comparativo_c0_vs_melhor": CAMINHO_GRAFICO_COMPARATIVO,
        "matrizes_confusao_fase1": CAMINHO_GRAFICO_MATRIZES,
        "importancia_hiperparametros_optuna": CAMINHO_GRAFICO_IMPORTANCIA,
        "fronteira_pareto_classificador": CAMINHO_GRAFICO_PARETO,
        "relatorio_json": CAMINHO_RELATORIO_FASE1,
    },
    "historico_trials": HISTORICO_TRIALS,
}

with open(CAMINHO_RELATORIO_FASE1, "w", encoding="utf-8") as arq:
    json.dump(relatorio_fase1, arq, ensure_ascii=False, indent=2)

print(f"Relatório completo da Fase 1 salvo com sucesso em: {CAMINHO_RELATORIO_FASE1}")